In [1]:
import pathlib
import sys

import duckdb
import pandas as pd

sys.path.insert(0, str(pathlib.Path('../src').resolve()))
import irp.config as _config

cfg = _config.load()
_ROOT = pathlib.Path(_config.__file__).parents[2]
DB = str((_ROOT / cfg['store']['db_path']).resolve())


def load_tables(*names: str) -> dict[str, pd.DataFrame]:
    with duckdb.connect(DB, read_only=True) as con:
        return {name: con.execute(f'SELECT * FROM "{name}"').df() for name in names}


print('DB:', DB)

DB: /mnt/Dev/active_python_projects/investment_research_platform/data/irp.duckdb


In [2]:
data = load_tables('income', 'balance', 'cashflow', 'companies', 'industries', 'sec_filings')
for name, df in data.items():
    print(f'{name:12s}: {len(df):,} rows')

income      : 68,495 rows
balance     : 68,486 rows
cashflow    : 68,486 rows
companies   : 6,556 rows
industries  : 74 rows
sec_filings : 68,493 rows


In [3]:
from irp.quality import run

findings = run(data)
print(f'Total findings: {len(findings):,}')
findings.head()

Total findings: 35,558


,table,ticker,company_name,isin,period,column,value,rule,detail,severity,edgar_url
0,balance,AMPS,"Altus Power, Inc.",US02217A1025,2021A,"Total Assets, Total Liabilities, Total Equity",0.032503,accounting_identity,"Assets=1113249000, Liab+Eq=1077064644, rel_err...",error,NaN
1,balance,OMEX,"Odyssey Marine Exploration, Inc.",US6761182012,2021A,"Total Assets, Total Liabilities, Total Equity",4.087908,accounting_identity,"Assets=8908887, Liab+Eq=45327595, rel_err=408.79%",error,https://www.sec.gov/Archives/edgar/data/798528...
2,balance,OMGA,"Omega Therapeutics, Inc.",US68217N1054,2020A,"Total Assets, Total Liabilities, Total Equity",0.011792,accounting_identity,"Assets=28083000, Liab+Eq=28414157, rel_err=1.18%",error,NaN
3,balance,PYCR,"Paycor HCM, Inc.",US70435P1021,2020A,"Total Assets, Total Liabilities, Total Equity",0.115771,accounting_identity,"Assets=2007783000, Liab+Eq=1775339879, rel_err...",error,NaN
4,balance,ROIC,Retail Opportunity Investments Corp.,US76131N1019,2021A,"Total Assets, Total Liabilities, Total Equity",0.02961,accounting_identity,"Assets=2928844000, Liab+Eq=2842122343, rel_err...",error,NaN


In [4]:
# Summary: findings per rule + severity
(
    findings.groupby(['severity', 'rule'])
    .size()
    .rename('count')
    .reset_index()
    .sort_values(['severity', 'count'], ascending=[True, False])
)

,severity,rule,count
1,error,impossible_value,259
0,error,accounting_identity,251
2,warning,sector_outlier,21059
3,warning,sudden_jump,13989


In [5]:
# Drill-down: accounting identity violations
findings[findings['rule'] == 'accounting_identity'].sort_values(
    'value', ascending=False
)

,table,ticker,company_name,isin,period,column,value,rule,detail,severity,edgar_url
1,balance,OMEX,"Odyssey Marine Exploration, Inc.",US6761182012,2021A,"Total Assets, Total Liabilities, Total Equity",4.087908,accounting_identity,"Assets=8908887, Liab+Eq=45327595, rel_err=408.79%",error,https://www.sec.gov/Archives/edgar/data/798528...
26,balance,OMEX,"Odyssey Marine Exploration, Inc.",US6761182012,2022A,"Total Assets, Total Liabilities, Total Equity",3.12382,accounting_identity,"Assets=13870827, Liab+Eq=57200799, rel_err=312...",error,https://www.sec.gov/Archives/edgar/data/798528...
30,balance,OMEX,"Odyssey Marine Exploration, Inc.",US6761182012,2020A,"Total Assets, Total Liabilities, Total Equity",2.570473,accounting_identity,"Assets=11759464, Liab+Eq=41986854, rel_err=257...",error,https://www.sec.gov/Archives/edgar/data/798528...
15,balance,OMEX,"Odyssey Marine Exploration, Inc.",US6761182012,2023A,"Total Assets, Total Liabilities, Total Equity",2.351466,accounting_identity,"Assets=22752297, Liab+Eq=76253552, rel_err=235...",error,https://www.sec.gov/Archives/edgar/data/798528...
10,balance,MTTR,"Matterport, Inc./DE",US5770961002,2020A,"Total Assets, Total Liabilities, Total Equity",2.28828,accounting_identity,"Assets=71852000, Liab+Eq=-92565481, rel_err=22...",error,NaN
...,...,...,...,...,...,...,...,...,...,...,...
160,balance,ENFN,"Enfusion, Inc.",US2928121043,2021Q3,"Total Assets, Total Liabilities, Total Equity",0.011269,accounting_identity,"Assets=49631000, Liab+Eq=50190272, rel_err=1.13%",error,NaN
238,balance,AAMC,Altisource Asset Management Corp,VI02153X1080,2023Q3,"Total Assets, Total Liabilities, Total Equity",0.011257,accounting_identity,"Assets=50001000, Liab+Eq=50563862, rel_err=1.13%",error,NaN
54,balance,BRST,"Broad Street Realty, Inc.",US1112941042,2021A,"Total Assets, Total Liabilities, Total Equity",0.010808,accounting_identity,"Assets=251669000, Liab+Eq=254389128, rel_err=1...",error,https://www.sec.gov/Archives/edgar/data/764897...
45,balance,SQSP,"Squarespace, Inc.",US85225A1079,2020A,"Total Assets, Total Liabilities, Total Equity",0.010289,accounting_identity,"Assets=306766000, Liab+Eq=309922462, rel_err=1...",error,NaN


In [6]:
# Drill-down: impossible values
findings[findings['rule'] == 'impossible_value']

,table,ticker,company_name,isin,period,column,value,rule,detail,severity,edgar_url
251,income,CNTB,Connect Biopharma Holdings Limited,US2075231017,2022A,Revenue,-4698000.0,impossible_value,Revenue is negative,error,NaN
252,income,TTE,TotalEnergies SE,US89151E1091,2021A,Revenue,-139851000000.0,impossible_value,Revenue is negative,error,NaN
253,income,ALT,"Altimmune, Inc.",US02155H2004,2022A,Revenue,-68000.0,impossible_value,Revenue is negative,error,https://www.sec.gov/Archives/edgar/data/132619...
254,income,PLUG,PLUG POWER INC,US72919P2020,2020A,Revenue,-93237000.0,impossible_value,Revenue is negative,error,https://www.sec.gov/Archives/edgar/data/109369...
255,income,REEMF,Rare Element Resources Ltd.,CA75381M1023,2023A,Revenue,-8242000.0,impossible_value,Revenue is negative,error,https://www.sec.gov/Archives/edgar/data/141980...
...,...,...,...,...,...,...,...,...,...,...,...
505,income,FBHS,Fortune Brands Home & Security,US34964C1062,2020Q4,Revenue,-809300000.0,impossible_value,Revenue is negative,error,NaN
506,income,MBOT,Microbot Medical Inc.,US59503A2042,2021Q3,Revenue,-3000.0,impossible_value,Revenue is negative,error,https://www.sec.gov/Archives/edgar/data/883975...
507,income,NTLA,"Intellia Therapeutics, Inc.",US45826J1051,2023Q4,Revenue,-1917000.0,impossible_value,Revenue is negative,error,https://www.sec.gov/Archives/edgar/data/165213...
508,income,VALPQ,Valaris plc,NaN,2021Q2,Revenue,-194600000.0,impossible_value,Revenue is negative,error,NaN


In [7]:
# Drill-down: sector outliers
findings[findings['rule'] == 'sector_outlier'].sort_values(
    'value', key=abs, ascending=False
)

,table,ticker,company_name,isin,period,column,value,rule,detail,severity,edgar_url
23750,income,JUPW,"Jupiter Wellness, Inc.",US48208F1057,2024Q4,Net Income,-236151.144,sector_outlier,"IQR-dist=-236151.1, sector=Healthcare, value=-...",warning,NaN
31666,income,JUPW,"Jupiter Wellness, Inc.",US48208F1057,2024Q4,Operating Income (Loss),-204008.443,sector_outlier,"IQR-dist=-204008.4, sector=Healthcare, value=-...",warning,NaN
35062,income,CRSP,CRISPR Therapeutics AG,CH0334081137,2021Q2,Operating Income (Loss),30297.497,sector_outlier,"IQR-dist=30297.5, sector=Healthcare, value=762...",warning,https://www.sec.gov/Archives/edgar/data/167441...
27503,income,CRSP,CRISPR Therapeutics AG,CH0334081137,2021Q2,Net Income,28508.039,sector_outlier,"IQR-dist=28508.0, sector=Healthcare, value=759...",warning,https://www.sec.gov/Archives/edgar/data/167441...
34871,income,CRSP,CRISPR Therapeutics AG,CH0334081137,2021Q4,Operating Income (Loss),-20612.093,sector_outlier,"IQR-dist=-20612.1, sector=Healthcare, value=-5...",warning,https://www.sec.gov/Archives/edgar/data/167441...
...,...,...,...,...,...,...,...,...,...,...,...
20379,income,GLW,CORNING INC /NY,US2193501051,2024A,Net Income,3.001,sector_outlier,"IQR-dist=3.0, sector=Technology, value=506000000",warning,https://www.sec.gov/Archives/edgar/data/24741/...
34507,income,TDG,TransDigm Group,US8936411003,2022Q2,Operating Income (Loss),3.001,sector_outlier,"IQR-dist=3.0, sector=Industrials, value=520000000",warning,https://www.sec.gov/Archives/edgar/data/126022...
33693,income,FSLR,"FIRST SOLAR, INC.",US3364331070,2023Q2,Operating Income (Loss),3.0,sector_outlier,"IQR-dist=3.0, sector=Technology, value=203970000",warning,https://www.sec.gov/Archives/edgar/data/127449...
25431,income,GEN,Gen Digital Inc.,NaN,2024Q2,Net Income,3.0,sector_outlier,"IQR-dist=3.0, sector=Technology, value=161000000",warning,NaN


In [8]:
# Drill-down: sudden jumps
findings[findings['rule'] == 'sudden_jump'].sort_values(
    'value', key=abs, ascending=False
)

,table,ticker,company_name,isin,period,column,value,rule,detail,severity,edgar_url
1331,income,JUPW,"Jupiter Wellness, Inc.",US48208F1057,2024Q4,Revenue,9669139.664,sudden_jump,Revenue: 110213 → 1065665000000 (966913966.4%),warning,NaN
7151,income,HTA,"HEALTHCARE TRUST OF AMERICA, INC.",US42225P5017,2020Q4,Net Income,2406497.3333,sudden_jump,Net Income: -30 → 72194890 (240649733.3%),warning,NaN
14068,balance,CRSP,CRISPR Therapeutics AG,CH0334081137,2021Q1,Total Assets,1071085.6613,sudden_jump,Total Assets: 1827966 → 1957910000000 (1071085...,warning,NaN
740,income,AVYA,Avaya Holdings Corp.,US05351X1019,2022Q2,Revenue,1004206.5736,sudden_jump,Revenue: 713 → 716000000 (100420657.4%),warning,NaN
13979,balance,AVYA,Avaya Holdings Corp.,US05351X1019,2022Q1,Total Assets,983624.731,sudden_jump,Total Assets: 5985 → 5887000000 (98362473.1%),warning,NaN
...,...,...,...,...,...,...,...,...,...,...,...
14125,balance,FNVT,Finnovate Acquisition Corp.,KYG3R34K1037,2024A,Total Assets,-0.8003,sudden_jump,Total Assets: 51237770 → 10233676 (-80.0%),warning,NaN
11377,income,SENEA,Seneca Foods Corporation,US8170705011,2022A,Net Income,-0.8002,sudden_jump,Net Income: 46200000 → 9231000 (-80.0%),warning,https://www.sec.gov/Archives/edgar/data/88948/...
1716,income,RKT,"Rocket Companies, Inc.",US77311W1018,2022Q1,Revenue,-0.8001,sudden_jump,Revenue: 7647193000 → 1529046000 (-80.0%),warning,NaN
2542,income,AMKR,"AMKOR TECHNOLOGY, INC.",US0316521006,2025Q1,Net Income,-0.8,sudden_jump,Net Income: 105649000 → 21128000 (-80.0%),warning,NaN


In [9]:
# Export all findings for manual review
out = pathlib.Path('../data/flagged_anomalies.csv')
findings.to_csv(out, index=False)
print(f'Exported {len(findings):,} findings → {out.resolve()}')

Exported 35,558 findings → /mnt/Dev/active_python_projects/investment_research_platform/data/flagged_anomalies.csv


## SEC Filings by Ticker

In [ ]:
import ipywidgets as widgets
from IPython.display import display

sec = data["sec_filings"].copy()
tickers = sorted(sec["ticker"].dropna().unique())

ticker_input = widgets.Combobox(
    options=tickers,
    placeholder="Type a ticker…",
    description="Ticker:",
    ensure_option=False,
    layout=widgets.Layout(width="220px"),
)
out = widgets.Output()


def _show(change):
    out.clear_output()
    t = ticker_input.value.strip().upper()
    if not t:
        return
    rows = sec[sec["ticker"] == t][["period", "url", "error"]].sort_values("period")
    with out:
        if rows.empty:
            print(f"No SEC filings found for {t!r}")
        else:
            display(rows.reset_index(drop=True).style.set_properties(**{"text-align": "left"}))


ticker_input.observe(_show, names="value")
display(ticker_input, out)

Combobox(value='', description='Ticker:', layout=Layout(width='220px'), options=('A', 'A21', 'AA', 'AAC', 'AAC…

Output()